In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_4.py ──
"""
Shared infrastructure for MLFP04 Exercise 4 — Anomaly Detection and Ensembles.

Contains: data loading (Singapore credit applications + an injected-anomaly
benchmark with an independent label), feature standardisation, score
normalisation helpers, metric reporting, visualisation shortcuts.

Technique-specific code (Z-score thresholding, Isolation Forest fit, LOF
neighbour count, blend weights, EnsembleEngine calls) does NOT belong here —
it lives in the per-technique files in `modules/mlfp04/solutions/ex_4/`.
"""

import asyncio
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

from kailash_ml import ExperimentTracker
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
np.random.seed(42)

OUTPUT_DIR = Path("outputs") / "ex4_anomaly"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA — Credit applications with an injected-anomaly benchmark
# ════════════════════════════════════════════════════════════════════════
# Why not just threshold a column and call it "fraud"? Because then the
# label is a function of an input feature and every AUC simply measures
# how much a detector looks at that one column (circular evaluation).
#
# Instead we follow the standard benchmark recipe for unsupervised anomaly
# detection (e.g. ADBench, Han et al., NeurIPS 2022): take REAL records as
# the normal population and inject anomalies of known TYPES, so the label
# comes from the injection process, never from a feature threshold:
#
#   global      — a real application with ONE field pushed 5-8 standard
#                 deviations beyond the mean (fat-finger entry, inflated
#                 declared balance). Extreme on a single feature.
#   dependency  — every field copied from a DIFFERENT real application
#                 (a "synthetic identity" stitched from real fragments).
#                 Each value is individually plausible; the COMBINATION is
#                 not (e.g. employment_years vs months_employed disagree).
#   clustered   — a tight group of near-identical applications (a
#                 coordinated application ring), shifted +3 std on three
#                 fields. Rare as a group, but dense locally.
#
# The normal rows are 20,000 real records from the course's Singapore
# credit-scoring dataset (mlfp02/sg_credit_scoring.parquet). Its real
# `default` outcome is kept (not as a feature) for a reality check:
# "statistically unusual" and "will default" are different questions.

N_NORMAL = 20_000
N_GLOBAL = 80
N_DEPENDENCY = 80
N_CLUSTERED = 40
ANOMALY_TYPES = ("global", "dependency", "clustered")
LABEL_COL = "is_anomaly"

# Numeric application fields with no nulls. Excluded on purpose:
#   income_sgd / loan_to_value — 30% / 65% missing
#   cpf_monthly_contribution   — near-constant (one value for most rows)
#   coe_vehicle_owner          — binary flag dominates Euclidean distance
#   future_default_indicator   — leaks the outcome
#   default                    — the outcome itself (reality check only)
FEATURE_COLS = [
    "age",
    "employment_years",
    "months_employed",
    "credit_utilization",
    "avg_balance_utilization",
    "num_credit_lines",
    "credit_age_years",
    "num_hard_inquiries",
    "payment_history_score",
    "num_late_payments",
    "revolving_balance",
    "installment_balance",
    "loan_amount_sgd",
    "monthly_installment",
    "num_dependents",
    "debt_to_income",
    "savings_balance",
    "checking_balance",
    "previous_defaults",
    "property_value_sgd",
]


def load_anomaly_frame(seed: int = 42) -> pl.DataFrame:
    """Return real credit applications plus injected, typed anomalies.

    Columns: FEATURE_COLS, `is_anomaly` (0/1), `anomaly_type`
    ("normal" / "global" / "dependency" / "clustered") and `default`
    (the real outcome for normal rows; null for injected rows). Rows are
    shuffled so injected anomalies are spread across the frame.
    """
    loader = MLFPDataLoader()
    raw = loader.load("mlfp02", "sg_credit_scoring.parquet")
    base = raw.sample(N_NORMAL, seed=seed).select(FEATURE_COLS + ["default"])
    B = base.select(FEATURE_COLS).to_numpy().astype(np.float64)
    mu, sd = B.mean(axis=0), B.std(axis=0)
    rng = np.random.default_rng(seed)
    n_feat = len(FEATURE_COLS)

    # global: one field pushed far into the upper tail
    G = B[rng.choice(len(B), N_GLOBAL, replace=False)].copy()
    for r in range(N_GLOBAL):
        j = int(rng.integers(n_feat))
        G[r, j] = mu[j] + rng.uniform(5.0, 8.0) * sd[j]

    # dependency: each field drawn from a different real application
    D = np.column_stack(
        [B[rng.integers(len(B), size=N_DEPENDENCY), j] for j in range(n_feat)]
    )

    # clustered: tight group around a shifted real application
    centre = B[int(rng.integers(len(B)))].copy()
    shifted = rng.choice(n_feat, 3, replace=False)
    centre[shifted] += 3.0 * sd[shifted]
    C = centre + rng.normal(0.0, 0.05, (N_CLUSTERED, n_feat)) * sd

    injected = np.vstack([G, D, C])
    types = ["global"] * N_GLOBAL + ["dependency"] * N_DEPENDENCY + [
        "clustered"
    ] * N_CLUSTERED

    normal_part = base.with_columns(
        pl.lit(0, dtype=pl.Int64).alias(LABEL_COL),
        pl.lit("normal").alias("anomaly_type"),
    )
    injected_part = pl.from_numpy(injected, schema=FEATURE_COLS).with_columns(
        pl.lit(None, dtype=pl.Int64).alias("default"),
        pl.lit(1, dtype=pl.Int64).alias(LABEL_COL),
        pl.Series("anomaly_type", types),
    )
    frame = pl.concat(
        [normal_part.with_columns(pl.col(FEATURE_COLS).cast(pl.Float64)), injected_part],
        how="vertical",
    )
    return frame.sample(fraction=1.0, shuffle=True, seed=seed)


def build_features(frame: pl.DataFrame) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Standardise FEATURE_COLS and return (X, y, feature_cols).

    Returns standardised X (float64) and the 0/1 `is_anomaly` label. The
    label is used ONLY for evaluation — every detector fits on X alone.
    """
    X, y, _col_info = to_sklearn_input(
        frame,
        feature_columns=FEATURE_COLS,
        target_column=LABEL_COL,
    )
    X_scaled = StandardScaler().fit_transform(X).astype(np.float64)
    return X_scaled, np.asarray(y).astype(int), list(FEATURE_COLS)


def load_dataset() -> tuple[np.ndarray, np.ndarray, list[str], pl.DataFrame]:
    """One-call helper: load the frame, build features, return everything."""
    frame = load_anomaly_frame()
    X, y, cols = build_features(frame)
    return X, y, cols, frame


def auc_by_type(frame: pl.DataFrame, scores: np.ndarray) -> dict[str, float]:
    """AUC-ROC of `scores` for each anomaly type vs the normal rows.

    Answers "WHICH kind of anomaly does this detector find?" — 1.0 means
    every anomaly of that type outranks every normal row; 0.5 is chance;
    below 0.5 means the detector ranks that type as MORE normal than the
    real applications.
    """
    types = frame["anomaly_type"].to_numpy()
    scores = np.asarray(scores, dtype=np.float64)
    out: dict[str, float] = {}
    for t in ANOMALY_TYPES:
        mask = (types == t) | (types == "normal")
        out[t] = float(roc_auc_score((types[mask] == t).astype(int), scores[mask]))
    return out


def print_auc_by_type(
    name: str, frame: pl.DataFrame, scores: np.ndarray
) -> dict[str, float]:
    """Compute per-type AUC, print it on one line, and return the dict."""
    by_type = auc_by_type(frame, scores)
    parts = "  ".join(f"{t}={v:.3f}" for t, v in by_type.items())
    print(f"  {name:<24} per-type AUC: {parts}")
    return by_type


def default_reality_check(frame: pl.DataFrame, scores: np.ndarray) -> float:
    """AUC of an anomaly score against the REAL `default` outcome.

    Computed on the real (non-injected) applications only.
    """
    mask = (frame["anomaly_type"] == "normal").to_numpy()
    defaults = frame["default"].to_numpy()[mask].astype(int)
    return float(roc_auc_score(defaults, np.asarray(scores)[mask]))


# ════════════════════════════════════════════════════════════════════════
# SCORE HELPERS
# ════════════════════════════════════════════════════════════════════════
# Anomaly detectors emit scores on wildly different scales. Normalising to
# [0, 1] (or to a rank) is what makes blending across methods possible.


def normalise_scores(scores: np.ndarray) -> np.ndarray:
    """Min-max normalise an anomaly score array to [0, 1]."""
    scores = np.asarray(scores, dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / (span + 1e-10)


def rank_normalise(scores: np.ndarray) -> np.ndarray:
    """Convert an anomaly score array to percentile ranks in [0, 1]."""
    from scipy.stats import rankdata

    return rankdata(np.asarray(scores, dtype=np.float64)) / len(scores)


def score_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    """Return AUC-ROC and average precision (AUC-PR) for an anomaly score."""
    return {
        "auc_roc": float(roc_auc_score(y_true, scores)),
        "avg_precision": float(average_precision_score(y_true, scores)),
    }


def print_metrics(
    name: str, y_true: np.ndarray, scores: np.ndarray
) -> dict[str, float]:
    """Compute metrics, print them on one line, and return the dict."""
    m = score_metrics(y_true, scores)
    print(f"  {name:<24} AUC-ROC={m['auc_roc']:.4f}  " f"AP={m['avg_precision']:.4f}")
    return m


def precision_at_recall(
    y_true: np.ndarray, scores: np.ndarray, target_recall: float
) -> tuple[float, float]:
    """Return (precision, threshold) at the TIGHTEST point where recall >= target.

    sklearn returns precisions/recalls ordered by ascending threshold, so
    recall decreases as threshold increases. We want the highest threshold
    that still meets the recall target — i.e. the last index where recall
    is still >= the target, which gives the maximum precision for that
    recall level.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, scores)
    # Drop the sentinel last point (precision=1.0, recall=0.0, no threshold)
    ps = precisions[:-1]
    rs = recalls[:-1]
    ts = thresholds
    mask = rs >= target_recall
    if not mask.any():
        return float(ps[0]), float(ts[0])
    # The tightest threshold satisfying the recall target is the largest
    # index where mask is True (thresholds are ascending).
    idx = int(np.where(mask)[0][-1])
    return float(ps[idx]), float(ts[idx])


def split_review_holdout(
    n_rows: int, review_fraction: float = 0.3, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Split row indices into a labelled 'review sample' and a holdout.

    Anything tuned with labels (blend weights, a supervised second stage)
    is fitted on the review sample and evaluated on the holdout, so the
    reported numbers are not scored on the rows that tuned them.
    """
    rng = np.random.default_rng(seed)
    order = rng.permutation(n_rows)
    cut = int(n_rows * review_fraction)
    return np.sort(order[:cut]), np.sort(order[cut:])


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION
# ════════════════════════════════════════════════════════════════════════


def write_comparison_chart(
    comparison: dict[str, dict[str, float]], filename: str
) -> Path:
    """Render a kailash-ml ModelVisualizer metric_comparison chart to HTML."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.metric_comparison(comparison)
    fig.update_layout(title="Anomaly Detection Method Comparison")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_roc_chart(
    y_true: np.ndarray, scores: np.ndarray, name: str, filename: str
) -> Path:
    """Render a ROC curve for a single detector."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.roc_curve(y_true, scores)
    fig.update_layout(title=f"ROC — {name}")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


def write_monitoring_chart(anomaly_rates: list[float], filename: str) -> Path:
    """Render an anomaly-rate-over-time chart for production monitoring."""
    from kailash_ml import ModelVisualizer

    viz = ModelVisualizer()
    fig = viz.training_history(
        {"Anomaly Rate %": [r * 100 for r in anomaly_rates]},
        x_label="Time Window",
    )
    fig.update_layout(title="Anomaly Rate Over Time (Production Monitoring)")
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    return path


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every anomaly technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_4 lesson logs its sweep + final-fit metrics to a single
# SQLite store so students can compare detectors across the lesson group
# (statistical, isolation forest, LOF, ensemble) after class. Mirrors the
# m4_clustering_zoo / m4_dimreduction_zoo experiments from ex_1 / ex_3.
#
# IMPORTANT: this store is SEPARATE from the clustering and dim-reduction
# stores. Per the SQLite contention trap (see .session-notes), running
# two ExperimentTracker writers against the same SQLite file concurrently
# triggers `disk I/O error`. Each exercise group gets its own DB.

ANOMALY_DB = "sqlite:///mlfp04_ex4_anomaly.db"
ANOMALY_EXPERIMENT_NAME = "m4_anomaly_zoo"


def _finite(x: float) -> float:
    """Tracker rejects NaN/inf via MetricValueError; coerce to 0.0.
    Anomaly metrics like AUC-ROC/AP can NaN on degenerate label arrays
    (e.g., all-positive or all-negative slices in a sweep)."""
    return float(x) if x == x and x not in (float("inf"), float("-inf")) else 0.0


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the anomaly-detection ExperimentTracker."""
    tracker = await ExperimentTracker.create(store_url=ANOMALY_DB)
    return tracker, ANOMALY_EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block. See ``rules/patterns.md`` § "Async Resource Cleanup".
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(name, float(value))
        if series_metrics:
            for name, values in series_metrics.items():
                for step, value in enumerate(values, start=1):
                    await run.log_metric(name, float(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 4.2: Isolation Forest Anomaly Detection
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Explain path-length isolation as an anomaly score
#   - Fit an Isolation Forest with the right contamination setting
#   - Sweep contamination to visualise the precision/recall trade-off
#   - Compare tree-based isolation against the statistical baselines
#
# PREREQUISITES: 4.1 (Z-score + IQR baselines).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — why "rare points get isolated faster"
#   2. Build — fit IsolationForest with a contamination sweep
#   3. Train — score every row with the best-performing fit
#   4. Visualise — ROC curve (written to outputs/)
#   5. Apply — synthetic-identity screening at a digital lender
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.ensemble import IsolationForest



### Kailash-ML ExperimentTracker — anomaly zoo shared store


In [ ]:
tracker, exp_name = setup_engines()

# Capture per-contamination sweep metrics so the TRACK section below can
# emit them after Task 5. Mutated inside the Task 2 sweep loop.
sweep_results: dict[float, dict[str, float]] = {}



## THEORY — Why Path Length Is an Anomaly Score


In [ ]:
# An Isolation Forest builds many random binary trees. At every node it
# picks a random feature and a random split value. A point gets "isolated"
# when a branch contains only that one point.
#
# Intuition: anomalies are rare AND far from the bulk of normal rows. A
# random split is MORE LIKELY to separate an anomaly from the rest of the
# data than to separate a crowded normal point. Anomalies end up at
# shallow leaves (few splits to isolate); normal points end up deep.
#
# Score: s(x) = 2 ^ (-E[h(x)] / c(n))
#     h(x)  = path length for x, averaged over all trees
#     c(n)  = average path length of an unsuccessful search in a BST
# Higher score = shorter path = more anomalous.
#
# PROS: scales to large N, handles high-dimensional features, no
# assumption of density or distribution. Parallelises across trees.
# CONS: the `contamination` parameter is an expert guess — set it from
# domain knowledge of the expected anomaly rate, NOT from the data.



## TASK 2 — BUILD: contamination sweep


In [ ]:
X, y, _feature_cols, frame = load_dataset()
n_samples, n_features = X.shape
print("\n" + "=" * 70)
print("  Isolation Forest Anomaly Detection")
print("=" * 70)
print(
    f"Rows: {n_samples:,} | Features: {n_features} | "
    f"Anomalies: {int(y.sum()):,} ({y.mean():.2%})"
)

print("\nContamination sweep:")
contamination_grid = [0.001, 0.005, 0.01, 0.02, 0.05]
for contam in contamination_grid:
    # TODO: Instantiate IsolationForest with n_estimators=200,
    # contamination=contam, random_state=42, n_jobs=-1
    model = ____

    # TODO: Call model.fit_predict(X) to get labels (-1 = anomaly, 1 = normal)
    preds = ____
    n_flagged = int((preds == -1).sum())
    flagged = preds == -1
    precision = float(y[flagged].mean()) if n_flagged else 0.0
    sweep_results[contam] = {
        "n_flagged": float(n_flagged),
        "precision": precision,
    }
    print(
        f"  contamination={contam:<6}  flagged={n_flagged:>5,}  "
        f"precision={precision:.3f}"
    )



## TASK 3 — TRAIN: fit the best-performing contamination


In [ ]:
# We pin contamination at 0.01 because ~1% of rows are injected anomalies.
# In production you would set this from domain knowledge of the expected
# anomaly rate, not from the label (which is unavailable at train time in
# a true anomaly detection setting). Note that contamination only moves
# the flag THRESHOLD — the anomaly scores (and so AUC) do not depend on it.

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1,
)
# TODO: Call iso_forest.fit(X)
____

# TODO: Extract anomaly scores. score_samples returns HIGHER for normal;
# negate so HIGHER means more anomalous.
iso_scores = ____
iso_labels = iso_forest.predict(X)

print("\nFinal Isolation Forest (contamination=0.01):")
iso_metrics = print_metrics("Isolation Forest", y, iso_scores)
print(f"  Predicted anomalies: {int((iso_labels == -1).sum()):,}")
print(f"  True anomalies:      {int(y.sum()):,}")

# Compare against the per-feature Z-score rule from 4.1, per anomaly type
# TODO: recompute the 4.1 max-|Z| score (X is already standardised)
z_scores = ____
print("\nPer anomaly type (1.0 = perfect, 0.5 = chance):")
iso_by_type = print_auc_by_type("Isolation Forest", frame, iso_scores)
z_by_type = print_auc_by_type("Z-score (4.1)", frame, z_scores)



### Checkpoint


In [ ]:
assert (
    iso_metrics["auc_roc"] > 0.5
), f"Isolation Forest AUC-ROC {iso_metrics['auc_roc']:.4f} should beat random"
assert iso_metrics["avg_precision"] > 0.0, "Isolation Forest AP should be positive"
assert (iso_labels == -1).sum() > 0, "Should flag at least one anomaly"
assert iso_scores.std() > 0, "Scores should vary across rows"
print("\n[ok] Checkpoint passed — Isolation Forest scored all rows\n")



## TASK 4 — VISUALISE: ROC curve


In [ ]:
roc_path = write_roc_chart(
    y, iso_scores, "Isolation Forest", "ex4_roc_isolation_forest.html"
)
print(f"Saved ROC chart: {roc_path}")

print("\nInterpretation (computed from the per-type AUCs above):")
for t in iso_by_type:
    diff = iso_by_type[t] - z_by_type[t]
    verdict = "better than" if diff > 0.02 else (
        "worse than" if diff < -0.02 else "about the same as"
    )
    print(
        f"  {t:<11} IF={iso_by_type[t]:.3f}  Z={z_by_type[t]:.3f}  "
        f"-> Isolation Forest is {verdict} the Z-score rule"
    )
print(
    "  Random splits use all features jointly, so IF can rank some"
    " 'dependency' rows (normal per feature, odd in combination) above"
    " chance where a per-feature rule cannot. On heavy-tailed real"
    " features, though, many genuine applications are also easy to"
    " isolate, so a single extreme field is not always ranked first."
)



## TASK 5 — APPLY: Synthetic-Identity Screening at a Digital Lender


In [ ]:
# SCENARIO (illustrative): a Singapore digital lender screens every new
# credit application overnight before approval. A known attack is the
# "synthetic identity" — an application assembled from fragments of
# several real people. Each field looks normal on its own (age, tenure,
# balances are all in range), but the joint combination is unusual
# (e.g. 30 years employed at age 25). That is exactly the 'dependency'
# anomaly type in this exercise's data.
#
# Why Isolation Forest is a reasonable first tool here:
#   - It scores every application on all 20 fields jointly, not one at
#     a time, so it can pick up some combination-level oddities
#   - Anomalous applications are rare (~1% here)
#   - It is fast: 200 trees on 20,000 rows fit in seconds on a laptop
#
# BUSINESS IMPACT (illustrative assumptions, not reported figures): if
# the lender reviews the top 1% most anomalous applications each night,
# the review load drops from every application to a few hundred, and
# each synthetic identity stopped before disbursement avoids the full
# loan amount. Use the per-type AUC above to judge how much of that
# benefit IF actually delivers on THIS data before promising it.
#
# LIMITATIONS: Isolation Forest is GLOBAL — it asks how easy a point is
# to separate from everything else. Points that are only odd relative
# to their LOCAL neighbourhood are LOF's territory (Exercise 4.3).



## TRACK — Log this lesson's run to the kailash-ml ExperimentTracker


In [ ]:
# Per-contamination sweep keys are formatted as `iso_contam_X_precision`
# / `iso_contam_X_flagged` where X is the contamination value with dots
# replaced by underscores (the tracker key regex disallows `.` followed
# by a digit, so 0.001 → "0_001"). Final-fit metrics use bare names.

sweep_metrics: dict[str, float] = {}
for contam, stats in sweep_results.items():
    label = str(contam).replace(".", "_")
    sweep_metrics[f"iso_contam_{label}_n_flagged"] = stats["n_flagged"]
    sweep_metrics[f"iso_contam_{label}_precision"] = _finite(stats["precision"])

# TODO: call track_run with run_name="isolation_forest". Headline scalars:
# iso_auc_roc + iso_avg_precision (from iso_metrics — wrap in _finite),
# iso_n_predicted_anomalies (count of iso_labels == -1). |-merge with
# sweep_metrics.
track_run(
    tracker,
    exp_name,
    run_name=____,
    params={
        "n_samples": n_samples,
        "n_features": n_features,
        "n_estimators": 200,
        "best_contamination": 0.01,
        "anomaly_rate": float(y.mean()),
    },
    scalar_metrics={
        "iso_auc_roc": _finite(iso_metrics["auc_roc"]),
        "iso_avg_precision": ____,
        "iso_n_predicted_anomalies": float(int((iso_labels == -1).sum())),
    }
    | sweep_metrics,
)
print(
    f"\n  [tracked] isolation_forest sweep + final-fit logged to {exp_name} "
    f"run='isolation_forest'\n"
)



## DESTINATION-FIRST CLOSE — AnomalyDetectionEngine.detect()


In [ ]:
# kailash-ml's AnomalyDetectionEngine.detect() wraps Isolation
# Forest with the same sklearn machinery this lesson hand-built. The
# engine handles polars→numpy conversion and emits an AnomalyResult
# (labels + scores + n_anomalies + algorithm-specific metrics) in one
# call. The tracker leaderboard is the natural follow-up.

import polars as pl

from kailash_ml.engines.anomaly_detection import AnomalyDetectionEngine

anomaly_df = pl.from_numpy(X, schema=_feature_cols)
# TODO: Instantiate AnomalyDetectionEngine and call .detect on anomaly_df
# with algorithm='isolation_forest' and contamination=0.01.
det = ____
fit_result = ____
fit_metrics = score_metrics(y, np.asarray(fit_result.scores))
print(
    f"  AnomalyDetectionEngine.detect(isolation_forest, contamination=0.01): "
    f"AUC-ROC={fit_metrics['auc_roc']:.4f}  "
    f"AP={fit_metrics['avg_precision']:.4f}  "
    f"n_anomalies={fit_result.n_anomalies}"
)
print(
    f"  Hand-rolled AUC-ROC (Task 3): {iso_metrics['auc_roc']:.4f} "
    f"— same algorithm under the hood, one-line vs ten-line interface.\n"
)



## REFLECTION


[x] Explained path-length isolation without opening the sklearn source
  [x] Ran a contamination sweep and read the precision trade-off
  [x] Fit IsolationForest on 20-feature tabular data with ~1% anomalies
  [x] Compared IF with the Z-score rule per anomaly type
  [x] Generated an ROC chart from ModelVisualizer
  [x] Framed a synthetic-identity screening scenario (illustrative figures)

  KEY INSIGHT: Isolation Forest scores points on all features JOINTLY,
  so it can see some combination-level anomalies that per-feature rules
  cannot. It is not uniformly better: on heavy-tailed real features a
  simple Z-score can rank single extreme values more sharply. Measure
  per anomaly type before choosing.

  Next: 03_local_outlier_factor.py — LOF compares LOCAL density, catching
  anomalies embedded in varying-density clusters.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)


# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

